# linkengine — quick start

A commented tour of **linkengine**: recognize, parse and normalize Italian legal citations
into stable identifiers (URN-NIR / ECLI / CELEX / PRAX). Every cell is runnable — read top to
bottom. Install the package first (`pip install -e .` from the repo root).

## 1 · Your first citation

`engine.extract(text)` returns one feature dict per recognized reference; `.rows[i]["urn"]`
is the canonical identifier.

In [ ]:
from linkengine import LinkEngine

engine = LinkEngine()
result = engine.extract("art. 2697 c.c.")
print("references:", len(result.rows))
result.rows[0]["urn"]

## 2 · What's inside a reference

A row is a plain `dict`: the canonical `urn` plus the recognition fields (`text`, `ref-type`,
`doc-type`, `number`, `year`, `partition`, …). Empty fields just don't apply to that reference.

In [ ]:
row = engine.extract("art. 43, comma 1, del d.P.R. n. 600 del 1973").rows[0]
{k: v for k, v in row.items() if v}        # show only the non-empty fields

## 3 · Several citations at once (segmentation)

Real text bundles many citations; the engine splits them into one row each (and expands
ranges like `artt. 15-18`).

In [ ]:
text = "Visto l'art. 2697 c.c. e la Cass. n. 100/2020, si applica il D.L. n. 34/2020."
for r in engine.extract(text).rows:
    print(f'{r["text"]:22} | {r["ref-type"]:12} | {r["urn"]}')

## 4 · The kinds of reference

Every reference resolves to one of four identifier schemes — `ref-type` tells you which.

In [ ]:
examples = {
    "legislation": "art. 19 del d.lgs. n. 546/1992",
    "caselaw":     "Cass. civ. n. 29036/2021",
    "prassi":      "Circolare AdE n. 25/E/2020",
    "EU act":      "direttiva 2006/112/CE",
}
for label, s in examples.items():
    r = engine.extract(s).rows[0]
    print(f'{label:12} | {r["ref-type"]:12} | {r["urn"]}')

## 5 · Identifier ⟷ text

`urn_to_text(urn)` is the inverse renderer — give it *only* an identifier.

In [ ]:
from linkengine import urn_to_text

for u in ["ECLI:IT:CASS:2020:1234CIV",
          "urn:nir:stato:regio.decreto:1942;262:2~art2697",
          "CELEX:32016R0679",
          "PRAX:AE:CIRC:2005:47"]:
    print(f"{u:48} -> {urn_to_text(u)}")

## 6 · Aliases, treaties, budget laws

Codes/consolidated texts (by name or abbreviation), bilateral tax treaties, and the annual
finanziaria / legge di bilancio all resolve to their underlying act.

In [ ]:
for s in ["art. 5 c.p.c.", "art. 109 TUIR", "art. 6 GDPR",
          "Convenzione Italia-Francia, art. 15", "legge di bilancio 2023"]:
    r = engine.extract(s).rows[0]
    print(f'{s:34} -> {r["urn"]}')

## 7 · Configuring context

Some references resolve only with context — pass defaults to the constructor (or to
`extract(...)` for one call).

In [ ]:
# self-reference: unresolved without a court, ECLI with one
print(engine.extract("questa Corte, sent. n. 50/2019").rows[0]["urn"] or "(unresolved)")
print(LinkEngine(default_authority="CORTE_CASS")
      .extract("questa Corte, sent. n. 50/2019").rows[0]["urn"])
# regional law without a region
s = "art. 5 della legge regionale n. 4 del 2007"
print(LinkEngine(default_region="Campania").extract(s).rows[0]["urn"])

## 8 · Under the hood

`extract(text, debug=True)` also fills `result.trace` (the spans each recognizer found) and
`result.references` (the character offsets of each citation).

In [ ]:
res = engine.extract("art. 43 del d.P.R. n. 600/1973", debug=True)
for module, spans in res.trace:
    for sp in spans:
        print(f"{module:12} {sp.entity.value:10} {sp.text!r}")

## 9 · HTML output — see what was recognized

`annotate_html(text)` wraps each reference in a tag carrying its fields; `render_html_document`
is a full styled page. Range partitions like `artt. 15-18` anchor the inner articles to the `-`.

In [ ]:
from linkengine import annotate_html, render_html_document
from IPython.display import HTML, display

text = "Visto l'art. 2697 c.c. e gli artt. 15-18 DPR 600/73; cfr. Cass. n. 100/2020."
print(annotate_html(text))
display(HTML(render_html_document(text)))

## 10 · Extending the library

The knowledge base is centralized: add a court in `catalog.py`, an alias in `aliases.py`, a
recognition pattern in `recognizers.py`. A quick look:

In [ ]:
from linkengine import catalog, aliases

print("courts :", len(catalog.COURTS), " e.g.", list(catalog.COURTS)[:6])
print("aliases:", len(aliases.ALIASES), " e.g.", [a.code for a in aliases.ALIASES[:6]])